# LOKI Chemical Tagging Toolkit

Converted from the provided Python script into a Google Colab-compatible notebook.


In [1]:
!pip install astropy galpy scikit-learn umap-learn numba matplotlib seaborn pandas numpy scipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.5 MB/s eta 0:00:00


In [ ]:
"""
================================================================================
LOKI CHEMICAL TAGGING & ASSEMBLY INDEX TOOLKIT
================================================================================
A comprehensive Python module for:
  1. Kinematic diagnostics of stellar populations (Galactic orbits, L_z, ecc)
  2. Chemical tagging via dimensionality reduction (t-SNE, UMAP)
  3. Single-system vs. multi-origin discrimination (MAD ratio test)
  4. Assembly Index (A_c) computation for halo substructure
  5. TNG-style synthetic merger debris generation
  6. White dwarf absence diagnostic & GCE mass estimation

Optimized for Google Colab / Jupyter Notebook usage.

Author: Cloud-9 Assembly Project
Date: 2026-05-03
================================================================================
"""

# ==============================================================================

## SECTION 0: SETUP & INSTALLS (Run this first in Colab)


In [2]:
# ==============================================================================
"""
!pip install astropy galpy scikit-learn umap-learn numba matplotlib seaborn pandas numpy scipy --quiet
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Astropy for coordinate transforms
from astropy.coordinates import SkyCoord, Galactocentric
from astropy import units as u
from astropy.table import Table

# Optional: galpy for orbit integration
try:
    from galpy.orbit import Orbit
    from galpy.potential import MWPotential2014, Potential
    GALPY_AVAILABLE = True
except ImportError:
    GALPY_AVAILABLE = False
    print("galpy not installed. Orbit integration will use approximate methods.")

# Optional: UMAP
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("umap-learn not installed. Only t-SNE will be available for dim reduction.")

# ==============================================================================

## SECTION 1: DATA LOADING & PREPROCESSING


In [3]:
# ==============================================================================

def load_stellar_catalog(filepath=None, df=None):
    """
    Load stellar catalog with kinematics and abundances.

    Expected columns (case-insensitive):
      - ra, dec: degrees
      - pmra, pmdec: mas/yr
      - radial_velocity, rv: km/s
      - distance, dist: pc or kpc (specify unit)
      - fe_h, [fe/h], feh: dex
      - mg_fe, ca_fe, ti_fe, cr_fe, mn_fe, ba_fe, sr_fe, eu_fe, etc.: dex
      - teff, logg: optional

    Parameters
    ----------
    filepath : str
        Path to CSV/FITS file
    df : pd.DataFrame
        Alternative: pass DataFrame directly

    Returns
    -------
    pd.DataFrame
        Standardized catalog
    """
    if df is None and filepath is not None:
        if filepath.endswith('.fits') or filepath.endswith('.fit'):
            from astropy.io import fits
            with fits.open(filepath) as hdul:
                df = Table(hdul[1].data).to_pandas()
        else:
            df = pd.read_csv(filepath)
    elif df is None:
        raise ValueError("Must provide either filepath or df")

    # Standardize column names
    df.columns = [c.lower().strip().replace('[', '').replace(']', '').replace('/', '_') for c in df.columns]

    # Map common variations
    col_map = {
        'fe_h': 'fe_h', 'feh': 'fe_h', 'fe': 'fe_h',
        'dist': 'distance', 'd': 'distance',
        'rv': 'radial_velocity', 'vr': 'radial_velocity',
        'pm_ra': 'pmra', 'pm_dec': 'pmdec'
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

    print(f"Loaded {len(df)} stars")
    print(f"Columns: {list(df.columns)}")
    return df


def compute_galactocentric_kinematics(df, distance_unit='pc',
                                      v_sun=(11.1, 245.0, 7.25),
                                      z_sun=20.8):
    """
    Convert observables to Galactocentric cylindrical coordinates and velocities.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain ra, dec, distance, pmra, pmdec, radial_velocity
    distance_unit : str
        'pc' or 'kpc'
    v_sun : tuple
        (U, V, W) of Sun in km/s (Schönrich+2010 defaults)
    z_sun : float
        Height of Sun above plane in pc

    Returns
    -------
    pd.DataFrame
        With added columns: x, y, z, R, phi, vR, vphi, vz, L_z, ecc, z_max
    """
    df = df.copy()

    # Distance in kpc
    if distance_unit == 'pc':
        dist_kpc = df['distance'].values / 1000.0
    else:
        dist_kpc = df['distance'].values

    # Create SkyCoord
    c = SkyCoord(
        ra=df['ra'].values * u.deg,
        dec=df['dec'].values * u.deg,
        distance=dist_kpc * u.kpc,
        pm_ra_cosdec=df['pmra'].values * u.mas/u.yr,
        pm_dec=df['pmdec'].values * u.mas/u.yr,
        radial_velocity=df['radial_velocity'].values * u.km/u.s
    )

    # Transform to Galactocentric
    galcen = c.transform_to(Galactocentric(
        galcen_distance=8.122 * u.kpc,
        z_sun=z_sun * u.pc,
        galcen_v_sun=v_sun * u.km/u.s
    ))

    # Cartesian
    df['x'] = galcen.x.to(u.kpc).value
    df['y'] = galcen.y.to(u.kpc).value
    df['z'] = galcen.z.to(u.kpc).value

    # Cylindrical
    df['R'] = np.sqrt(df['x']**2 + df['y']**2)
    df['phi'] = np.arctan2(df['y'], df['x'])  # radians

    # Velocities in cylindrical coords
    v_x = galcen.v_x.to(u.km/u.s).value
    v_y = galcen.v_y.to(u.km/u.s).value
    v_z = galcen.v_z.to(u.km/u.s).value

    df['vR'] = (df['x'] * v_x + df['y'] * v_y) / df['R']
    df['vphi'] = (-df['y'] * v_x + df['x'] * v_y) / df['R']
    df['vz'] = v_z

    # Angular momentum L_z = R * vphi
    df['L_z'] = df['R'] * df['vphi']  # kpc km/s

    # Prograde vs Retrograde
    df['orbit_sense'] = np.where(df['L_z'] > 0, 'prograde', 'retrograde')

    # Approximate eccentricity and z_max using energy conservation
    # (Simplified; use galpy for precise values)
    v_tot = np.sqrt(df['vR']**2 + df['vphi']**2 + df['vz']**2)

    # Simple potential approximation: Phi(R,z) ~ v_c^2 * ln(R) + 0.5 * v_c^2 * (z/R_d)^2
    v_c = 220.0  # km/s
    R_d = 3.0    # kpc disk scale length

    Phi = v_c**2 * np.log(df['R']) + 0.5 * v_c**2 * (df['z'] / R_d)**2
    E = 0.5 * v_tot**2 + Phi
    L_z_mag = np.abs(df['L_z'])

    # Circular velocity at R
    v_circ = v_c

    # Approximate eccentricity
    df['ecc'] = np.abs(df['vR']) / v_tot  # Simplified proxy
    df['z_max'] = np.abs(df['z']) * (1 + 0.5 * (df['vz'] / v_c)**2)  # Approximate

    return df


def filter_loki_candidates(df, fe_h_max=-2.0, z_max_max=4.0, ecc_min=0.5, R_max=15.0):
    """
    Apply Loki selection criteria from the MNRAS paper.

    Parameters
    ----------
    df : pd.DataFrame
        With fe_h, z_max, ecc, R columns
    fe_h_max : float
        Maximum [Fe/H] (default -2.0 for VMP)
    z_max_max : float
        Maximum Z_max in kpc (default 4.0)
    ecc_min : float
        Minimum eccentricity (default 0.5)
    R_max : float
        Maximum Galactocentric radius in kpc

    Returns
    -------
    pd.DataFrame
        Filtered candidate sample
    """
    mask = (
        (df['fe_h'] <= fe_h_max) &
        (df['z_max'] <= z_max_max) &
        (df['ecc'] >= ecc_min) &
        (df['R'] <= R_max)
    )
    filtered = df[mask].copy()
    print(f"Loki candidates: {len(filtered)} / {len(df)} stars")
    print(f"  Prograde: {sum(filtered['L_z'] > 0)}")
    print(f"  Retrograde: {sum(filtered['L_z'] < 0)}")
    return filtered


# ==============================================================================

## SECTION 2: CHEMICAL TAGGING & DIMENSIONALITY REDUCTION


In [4]:
# ==============================================================================

def prepare_abundance_matrix(df, abundance_cols=None, scale=True):
    """
    Extract and optionally scale abundance data for clustering/dim reduction.

    Parameters
    ----------
    df : pd.DataFrame
    abundance_cols : list
        List of column names for [X/Fe] or [X/H] ratios
        Default: common alpha, iron-peak, neutron-capture elements
    scale : bool
        Whether to StandardScale the matrix

    Returns
    -------
    np.ndarray, list
        Abundance matrix (n_stars x n_elements), column names
    """
    if abundance_cols is None:
        # Default set matching Loki paper analysis
        default_cols = ['mg_fe', 'ca_fe', 'ti_fe', 'cr_fe', 'mn_fe',
                        'co_fe', 'ni_fe', 'sr_fe', 'ba_fe', 'eu_fe',
                        'si_fe', 'al_fe', 'c_fe', 'n_fe', 'o_fe']
        abundance_cols = [c for c in default_cols if c in df.columns]
        if len(abundance_cols) == 0:
            # Fallback: auto-detect [X/Fe] columns
            abundance_cols = [c for c in df.columns if c.endswith('_fe') and c != 'fe_h']

    X = df[abundance_cols].values

    # Handle NaNs
    col_means = np.nanmedian(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])

    if scale:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

    return X, abundance_cols


def run_chemical_tagging(df, abundance_cols=None, method='tsne',
                         perplexity=5, n_neighbors=5, min_dist=0.1,
                         random_state=42):
    """
    Perform chemical tagging via dimensionality reduction.

    Parameters
    ----------
    df : pd.DataFrame
    abundance_cols : list
    method : str
        'tsne', 'umap', or 'pca'
    perplexity : int
        t-SNE perplexity (small for small samples like Loki n=20)
    n_neighbors : int
        UMAP n_neighbors
    min_dist : float
        UMAP min_dist

    Returns
    -------
    pd.DataFrame
        Original df with 'chem_x', 'chem_y' columns added
    """
    X, cols_used = prepare_abundance_matrix(df, abundance_cols)

    if method == 'tsne':
        reducer = TSNE(n_components=2, perplexity=min(perplexity, len(X)-1),
                       random_state=random_state, n_iter=1000)
        embedding = reducer.fit_transform(X)
    elif method == 'umap' and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, n_neighbors=min(n_neighbors, len(X)-1),
                           min_dist=min_dist, random_state=random_state)
        embedding = reducer.fit_transform(X)
    else:
        from sklearn.decomposition import PCA
        reducer = PCA(n_components=2)
        embedding = reducer.fit_transform(X)

    df = df.copy()
    df['chem_x'] = embedding[:, 0]
    df['chem_y'] = embedding[:, 1]
    df['chem_method'] = method

    return df


def mad_ratio_test(df, abundance_cols=None, group_col='orbit_sense'):
    """
    Compute Median Absolute Deviation (MAD) ratio test for single vs. multi-origin.

    From the Loki paper: if prograde and retrograde subsamples have similar MADs
    to the full sample, they likely share a single origin.

    Parameters
    ----------
    df : pd.DataFrame
    abundance_cols : list
    group_col : str
        Column to split groups (e.g., 'orbit_sense')

    Returns
    -------
    dict
        MAD statistics and ratios
    """
    X, cols = prepare_abundance_matrix(df, abundance_cols, scale=False)

    def compute_mad(data):
        med = np.median(data, axis=0)
        return np.median(np.abs(data - med), axis=0)

    mad_full = compute_mad(X)

    results = {
        'full_mad_mean': np.mean(mad_full),
        'full_mad_per_element': dict(zip(cols, mad_full)),
        'groups': {}
    }

    for group_name, group_df in df.groupby(group_col):
        X_g, _ = prepare_abundance_matrix(group_df, cols, scale=False)
        if len(X_g) < 3:
            continue
        mad_g = compute_mad(X_g)
        ratio = np.mean(mad_g) / np.mean(mad_full)
        results['groups'][group_name] = {
            'n_stars': len(X_g),
            'mad_mean': np.mean(mad_g),
            'mad_ratio_to_full': ratio,
            'mad_per_element': dict(zip(cols, mad_g))
        }

    # Interpretation
    ratios = [v['mad_ratio_to_full'] for v in results['groups'].values()]
    if ratios:
        results['interpretation'] = (
            f"MAD ratios: {dict(zip(results['groups'].keys(), [f'{r:.2f}' for r in ratios]))}. "
            f"Values near 1.0 suggest single origin; << 1 suggests distinct groups."
        )

    return results


# ==============================================================================

## SECTION 3: ASSEMBLY INDEX (A_c) FOR SUBSTRUCTURE


In [5]:
# ==============================================================================

class AssemblyIndexCalculator:
    """
    Compute Cosmological Assembly Index (A_c) for stellar populations.

    Based on the Cloud-9 framework: incorporates chemical entropy,
    kinematic complexity, and topological structure.
    """

    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def chemical_entropy(self, df, abundance_cols=None):
        """
        Shannon entropy of the chemical distribution.
        Lower entropy = more chemically coherent = higher A_c potential.
        """
        X, _ = prepare_abundance_matrix(df, abundance_cols, scale=True)

        # Use k-NN density to estimate entropy
        from sklearn.neighbors import KernelDensity
        kde = KernelDensity(bandwidth=0.5, kernel='gaussian')
        kde.fit(X)
        log_dens = kde.score_samples(X)

        # Entropy estimate
        entropy = -np.mean(log_dens)
        return entropy

    def kinematic_complexity(self, df):
        """
        Measure orbital complexity via velocity dispersion tensor anisotropy.
        """
        vels = df[['vR', 'vphi', 'vz']].values
        cov = np.cov(vels.T)
        eigenvals = np.sort(np.linalg.eigvalsh(cov))[::-1]

        # Anisotropy parameter beta = 1 - (sigma_theta^2 + sigma_phi^2)/(2 sigma_r^2)
        # Simplified: use eigenvalue ratios
        if eigenvals[0] > 0:
            anisotropy = 1 - (eigenvals[1] + eigenvals[2]) / (2 * eigenvals[0])
        else:
            anisotropy = 0

        # Complexity = normalized sum of eigenvalues (total dispersion) * anisotropy
        complexity = np.sqrt(np.sum(eigenvals)) * (1 + abs(anisotropy))
        return complexity, anisotropy

    def topological_complexity(self, df, abundance_cols=None):
        """
        Persistent homology-inspired topological complexity measure.
        Simplified: use nearest-neighbor distance distribution skewness.
        Clustered populations have bimodal NN-distance distributions.
        """
        X, _ = prepare_abundance_matrix(df, abundance_cols, scale=True)

        if len(X) < self.n_neighbors + 1:
            return 0.0

        nbrs = NearestNeighbors(n_neighbors=self.n_neighbors+1).fit(X)
        distances, _ = nbrs.kneighbors(X)
        # Exclude self-distance
        nn_distances = distances[:, 1:].mean(axis=1)

        # Skewness of log NN distances
        log_d = np.log(nn_distances + 1e-10)
        skewness = stats.skew(log_d)

        # High |skewness| suggests substructure (clusters + field)
        return abs(skewness)

    def integrated_information(self, df, abundance_cols=None):
        """
        Proxy for integrated information (phi) via mutual information
        between chemical and kinematic spaces.
        """
        X_chem, _ = prepare_abundance_matrix(df, abundance_cols, scale=True)
        X_kin = df[['R', 'L_z', 'ecc', 'z_max']].values

        # Standardize
        X_chem = StandardScaler().fit_transform(X_chem)
        X_kin = StandardScaler().fit_transform(X_kin)

        # Canonical correlation analysis proxy
        from sklearn.cross_decomposition import CCA
        n_comp = min(2, X_chem.shape[1], X_kin.shape[1])
        cca = CCA(n_components=n_comp)
        cca.fit(X_chem, X_kin)
        X_c, X_k = cca.transform(X_chem, X_kin)

        # Correlation between canonical variables
        mi_proxy = 0
        for i in range(n_comp):
            mi_proxy += np.corrcoef(X_c[:, i], X_k[:, i])[0, 1]**2

        return mi_proxy / n_comp

    def compute_ac(self, df, abundance_cols=None, weights=None):
        """
        Compute Assembly Index A_c.

        A_c = w1 * (1/S_chem) + w2 * K_complex + w3 * T_complex + w4 * I_info

        Higher A_c = more structured, non-random, likely accreted substructure.

        Parameters
        ----------
        df : pd.DataFrame
            Stellar population
        abundance_cols : list
        weights : dict
            Optional custom weights

        Returns
        -------
        dict
            Full A_c decomposition
        """
        if weights is None:
            weights = {
                'chemical': 0.3,
                'kinematic': 0.25,
                'topological': 0.25,
                'information': 0.2
            }

        S_chem = self.chemical_entropy(df, abundance_cols)
        K_comp, aniso = self.kinematic_complexity(df)
        T_comp = self.topological_complexity(df, abundance_cols)
        I_info = self.integrated_information(df, abundance_cols)

        # Normalize components (approximate scaling)
        A_c = (
            weights['chemical'] * (1.0 / (1.0 + S_chem)) +
            weights['kinematic'] * np.tanh(K_comp / 100) +
            weights['topological'] * np.tanh(T_comp / 2) +
            weights['information'] * I_info
        )

        return {
            'A_c': A_c,
            'chemical_entropy': S_chem,
            'kinematic_complexity': K_comp,
            'anisotropy': aniso,
            'topological_complexity': T_comp,
            'integrated_information': I_info,
            'weights': weights,
            'n_stars': len(df)
        }


# ==============================================================================

## SECTION 4: WHITE DWARF ABSENCE DIAGNOSTIC & GCE MASS ESTIMATION


In [6]:
# ==============================================================================

def white_dwarf_absence_test(df, sn_ia_indicators=None):
    """
    Test for absence of Type Ia supernova / white dwarf explosion signatures.

    In the Loki paper, the absence of [Mn/Fe] enhancement and other SN Ia
    tracers suggests the progenitor was too short-lived to form white dwarfs.

    Parameters
    ----------
    df : pd.DataFrame
    sn_ia_indicators : list
        Columns that trace SN Ia (default: ['mn_fe', 'ni_fe'])

    Returns
    -------
    dict
        Diagnostic results
    """
    if sn_ia_indicators is None:
        sn_ia_indicators = ['mn_fe', 'ni_fe']

    available = [c for c in sn_ia_indicators if c in df.columns]
    if not available:
        return {'error': 'No SN Ia indicator columns found'}

    results = {}
    for col in available:
        mean_val = df[col].mean()
        std_val = df[col].std()
        results[col] = {
            'mean': mean_val,
            'std': std_val,
            'suggestive_of_snia': mean_val > 0.0  # Simplified threshold
        }

    # Overall assessment
    snia_present = any(results[c]['suggestive_of_snia'] for c in available)
    results['assessment'] = (
        "SN Ia signatures DETECTED" if snia_present
        else "SN Ia signatures ABSENT — consistent with short-lived progenitor (Loki-like)"
    )

    return results


def estimate_progenitor_mass(df, yield_model='simple',
                             fe_yield_ccsn=0.07, m_fe_star=1e-4):
    """
    Estimate baryonic mass of progenitor dwarf galaxy from chemical yields.

    Simplified GCE model: M_baryon ~ M_Fe_total / (fe_yield * f_retention)

    Parameters
    ----------
    df : pd.DataFrame
    yield_model : str
        'simple' or 'detailed'
    fe_yield_ccsn : float
        Iron yield per CCSN in solar masses
    m_fe_star : float
        Average Fe mass per star in solar masses

    Returns
    -------
    dict
        Mass estimates
    """
    n_stars = len(df)
    fe_h_mean = df['fe_h'].mean()

    # Total iron mass in stars (relative to solar)
    fe_mass_total = n_stars * m_fe_star * 10**fe_h_mean

    # Number of CCSNe required
    n_ccsn = fe_mass_total / fe_yield_ccsn

    # Assume IMF: 1 CCSN per ~100 M_sun of star formation
    m_star_formed = n_ccsn * 100

    # Gas retention factor (dwarfs retain ~10-30%)
    f_ret = 0.2
    m_baryon = m_star_formed / f_ret

    return {
        'n_stars': n_stars,
        'mean_fe_h': fe_h_mean,
        'fe_mass_stars': fe_mass_total,
        'n_ccsn_required': n_ccsn,
        'm_star_formed': m_star_formed,
        'm_baryon_estimate': m_baryon,
        'note': 'Simplified model. For accurate results, use StarFit or full GCE code.'
    }


# ==============================================================================

## SECTION 5: TNG SYNTHETIC MERGER DEBRIS GENERATOR


In [7]:
# ==============================================================================

def generate_synthetic_merger_debris(n_stars=100,
                                     progenitor_mass=1e8,
                                     accretion_time=12.0,  # Gyr ago
                                     baryon_fraction=0.1,
                                     random_state=42):
    """
    Generate synthetic stellar debris from an accreted dwarf galaxy.

    Simplified model for testing detection algorithms.

    Parameters
    ----------
    n_stars : int
        Number of synthetic stars
    progenitor_mass : float
        Baryonic mass in solar masses
    accretion_time : float
        Lookback time of accretion in Gyr
    baryon_fraction : float

    Returns
    -------
    pd.DataFrame
        Synthetic stellar catalog
    """
    rng = np.random.RandomState(random_state)

    # Metallicity: metal-poor, with some dispersion
    fe_h = rng.normal(-2.5, 0.3, n_stars)
    fe_h = np.clip(fe_h, -4.0, -1.5)

    # Alpha enhancement
    mg_fe = 0.4 + 0.1 * rng.randn(n_stars)
    ca_fe = 0.3 + 0.1 * rng.randn(n_stars)
    ti_fe = 0.3 + 0.1 * rng.randn(n_stars)

    # No SN Ia products
    mn_fe = -0.3 + 0.05 * rng.randn(n_stars)
    ni_fe = -0.2 + 0.05 * rng.randn(n_stars)

    # Kinematics: bimodal L_z (prograde/retrograde from chaotic early merger)
    n_prog = rng.binomial(n_stars, 0.55)
    n_retro = n_stars - n_prog

    # Prograde: L_z ~ 500-1500 kpc km/s
    lz_prog = rng.uniform(500, 1500, n_prog)
    # Retrograde: L_z ~ -1500 to -500
    lz_retro = rng.uniform(-1500, -500, n_retro)

    L_z = np.concatenate([lz_prog, lz_retro])
    rng.shuffle(L_z)  # Mix them

    # R, z, ecc
    R = rng.uniform(2, 12, n_stars)
    z = rng.normal(0, 1.5, n_stars)
    ecc = rng.uniform(0.5, 0.95, n_stars)

    # vR, vz from L_z and ecc
    vphi = L_z / R
    vR = np.sqrt((220**2 * ecc**2) / (1 - ecc**2 + 0.1))  # Approximate
    vR *= np.sign(rng.randn(n_stars))  # Random signs
    vz = z * rng.uniform(20, 60, n_stars)  # km/s

    # Cartesian positions (approximate)
    phi = rng.uniform(0, 2*np.pi, n_stars)
    x = R * np.cos(phi)
    y = R * np.sin(phi)

    df = pd.DataFrame({
        'fe_h': fe_h,
        'mg_fe': mg_fe,
        'ca_fe': ca_fe,
        'ti_fe': ti_fe,
        'mn_fe': mn_fe,
        'ni_fe': ni_fe,
        'x': x, 'y': y, 'z': z,
        'R': R,
        'L_z': L_z,
        'vR': vR,
        'vphi': vphi,
        'vz': vz,
        'ecc': ecc,
        'z_max': np.abs(z) * 1.5,
        'orbit_sense': np.where(L_z > 0, 'prograde', 'retrograde'),
        'source': 'synthetic_loki'
    })

    return df


# ==============================================================================

## SECTION 6: VISUALIZATION SUITE


In [ ]:
# ==============================================================================

def plot_chemical_tagging(df, color_by='orbit_sense', save_path=None):
    """
    Plot chemical tagging embedding with kinematic overlays.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Chemical space colored by orbit sense
    ax = axes[0]
    for sense, group in df.groupby(color_by):
        ax.scatter(group['chem_x'], group['chem_y'], label=sense, s=80, alpha=0.8, edgecolors='k')
    ax.set_xlabel('Chemical Dim 1')
    ax.set_ylabel('Chemical Dim 2')
    ax.set_title(f'Chemical Tagging ({df["chem_method"].iloc[0]})')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 2: Toomre diagram
    ax = axes[1]
    vtot = np.sqrt(df['vR']**2 + df['vz']**2)
    ax.scatter(df['vphi'], vtot, c=df['fe_h'], cmap='viridis_r', s=80, edgecolors='k')
    ax.axvline(0, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel(r'$v_\phi$ [km/s]')
    ax.set_ylabel(r'$\sqrt{v_R^2 + v_z^2}$ [km/s]')
    ax.set_title('Toomre Diagram')
    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label('[Fe/H]')
    ax.grid(True, alpha=0.3)

    # Plot 3: L_z vs R
    ax = axes[2]
    scatter = ax.scatter(df['R'], df['L_z'], c=df['ecc'], cmap='plasma', s=80, edgecolors='k')
    ax.axhline(0, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('R [kpc]')
    ax.set_ylabel(r'$L_z$ [kpc km/s]')
    ax.set_title('Angular Momentum vs Radius')
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Eccentricity')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_abundance_patterns(df, abundance_cols=None, group_col='orbit_sense'):
    """
    Plot abundance pattern comparison between groups.
    """
    if abundance_cols is None:
        abundance_cols = [c for c in df.columns if c.endswith('_fe') and c != 'fe_h']

    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(abundance_cols))
    width = 0.35

    for i, (sense, group) in enumerate(df.groupby(group_col)):
        means = [group[col].mean() for col in abundance_cols]
        stds = [group[col].std() for col in abundance_cols]
        offset = width * (i - 0.5)
        ax.bar(x + offset, means, width, yerr=stds, label=sense, alpha=0.8, capsize=3)

    ax.set_xticks(x)
    ax.set_xticklabels(abundance_cols, rotation=45, ha='right')
    ax.set_ylabel('[X/Fe]')
    ax.set_title('Abundance Patterns by Orbital Sense')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(0, color='k', linestyle='-', linewidth=0.5)

    plt.tight_layout()
    plt.show()


# ==============================================================================

## SECTION 7: FULL PIPELINE EXAMPLE


In [10]:
# ==============================================================================

def run_loki_pipeline(df, abundance_cols=None, run_ac=True):
    """
    Run the complete Loki analysis pipeline.

    Parameters
    ----------
    df : pd.DataFrame
        Input stellar catalog
    abundance_cols : list
        Abundance columns to use
    run_ac : bool
        Whether to compute Assembly Index

    Returns
    -------
    dict
        Complete results
    """
    print("="*60)
    print("LOKI ANALYSIS PIPELINE")
    print("="*60)

    # Step 1: Kinematics
    print("\n[1/6] Computing Galactocentric kinematics...")
    if 'x' not in df.columns:
        df = compute_galactocentric_kinematics(df)
    else:
        print("  (Kinematics already present)")

    # Step 2: Filter candidates
    print("\n[2/6] Filtering Loki candidates...")
    candidates = filter_loki_candidates(df)

    if len(candidates) == 0:
        print("  No candidates found. Aborting.")
        return None

    # Step 3: Chemical tagging
    print("\n[3/6] Running chemical tagging (t-SNE)...")
    candidates = run_chemical_tagging(candidates, abundance_cols, method='tsne')

    # Step 4: MAD ratio test
    print("\n[4/6] Computing MAD ratio test...")
    mad_results = mad_ratio_test(candidates, abundance_cols)
    print(f"  {mad_results['interpretation']}")

    # Step 5: White dwarf absence
    print("\n[5/6] White dwarf absence diagnostic...")
    wd_results = white_dwarf_absence_test(candidates)
    print(f"  {wd_results['assessment']}")

    # Step 6: Assembly Index
    ac_results = None
    if run_ac:
        print("\n[6/6] Computing Assembly Index...")
        calc = AssemblyIndexCalculator()
        ac_results = calc.compute_ac(candidates, abundance_cols)
        print(f"  A_c = {ac_results['A_c']:.3f}")
        print(f"  Chemical entropy: {ac_results['chemical_entropy']:.3f}")
        print(f"  Kinematic complexity: {ac_results['kinematic_complexity']:.1f}")
        print(f"  Topological complexity: {ac_results['topological_complexity']:.3f}")
        print(f"  Integrated information: {ac_results['integrated_information']:.3f}")

    # Step 7: Progenitor mass
    print("\n[Bonus] Estimating progenitor mass...")
    mass_results = estimate_progenitor_mass(candidates)
    print(f"  Estimated baryonic mass: {mass_results['m_baryon_estimate']:.2e} M_sun")

    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)

    return {
        'candidates': candidates,
        'mad_test': mad_results,
        'white_dwarf_test': wd_results,
        'assembly_index': ac_results,
        'progenitor_mass': mass_results
    }


# ==============================================================================
# COLAB DEMO: Generate synthetic data and run full analysis
# ==============================================================================
if __name__ == "__main__":
    print("Generating synthetic Loki-like population...")
    synthetic_df = generate_synthetic_merger_debris(n_stars=50, random_state=42)

    print("\nRunning full pipeline on synthetic data...")
    results = run_loki_pipeline(synthetic_df)

    if results:
        print("\nGenerating plots...")
        plot_chemical_tagging(results['candidates'])
        plot_abundance_patterns(results['candidates'])

Generating synthetic Loki-like population...

Running full pipeline on synthetic data...
LOKI ANALYSIS PIPELINE

[1/6] Computing Galactocentric kinematics...
  (Kinematics already present)

[2/6] Filtering Loki candidates...
Loki candidates: 42 / 50 stars
  Prograde: 27
  Retrograde: 15

[3/6] Running chemical tagging (t-SNE)...

[4/6] Computing MAD ratio test...
  MAD ratios: {'prograde': '1.08', 'retrograde': '0.91'}. Values near 1.0 suggest single origin; << 1 suggests distinct groups.

[5/6] White dwarf absence diagnostic...
  SN Ia signatures ABSENT — consistent with short-lived progenitor (Loki-like)

[6/6] Computing Assembly Index...
  A_c = 0.367
  Chemical entropy: 4.723
  Kinematic complexity: 542.3
  Topological complexity: 0.259
  Integrated information: 0.160

[Bonus] Estimating progenitor mass...
  Estimated baryonic mass: 8.25e-02 M_sun

PIPELINE COMPLETE

Generating plots...


NameError: name 'plot_chemical_tagging' is not defined